<div style="background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 5</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Log Compaction</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Understand how Kafka retains the latest state using log compaction.</p>
</div>

---

## 🎯 Overview

Kafka supports different cleanup policies for its topics. The default is `delete`, which drops old data based on time or size. In this lab, we will explore the `compact` policy.

- 🗜️ **Compaction (`cleanup.policy=compact`):** Retains only the *latest* value for each unique key. This is perfect for database change logs (CDC) or tracking user state, where you only care about the most recent information.

---

## ⚙️ Prerequisites

We will use the standard single-node cluster from the root directory for this lab.

<div style="background-color: rgba(243, 156, 18, 0.1); border-left: 4px solid #f39c12; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>⚠️ Important:</strong> Ensure any previous multi-broker clusters are stopped. Run <code>docker-compose down</code> in Lab4 before proceeding.
</div>

---

## <span style="color: #11998e;">Step 1:</span> Start the Kafka Cluster


In [ ]:
!docker-compose -f ../../docker-compose.yml up -d

---

## <span style="color: #11998e;">Step 2:</span> Create a Compacted Topic

We will create a topic named `user-balances` specifically configured for log compaction. We set aggressive compaction settings (`segment.ms` and `min.cleanable.dirty.ratio`) so that the background cleaner thread triggers quickly for our demonstration.

In [ ]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --create \
  --topic user-balances \
  --partitions 1 \
  --replication-factor 1 \
  --config cleanup.policy=compact \
  --config segment.ms=100 \
  --config min.cleanable.dirty.ratio=0.01

---

## <span style="color: #11998e;">Step 3:</span> Produce Data with Keys

Produce some data with keys. Notice how we are updating the balance for `user1` and `user2` over time.
*(We use `%%writefile` to create a text file, copy it into the container, and pipe it into the producer).*

In [ ]:
%%writefile balances.txt
user1:100
user2:150
user1:120
user3:50
user2:80
user1:200

In [ ]:
# Copy the file into the broker container
!docker cp balances.txt kafka:/tmp/balances.txt

# Produce the messages
!docker exec kafka bash -c "kafka-console-producer --bootstrap-server localhost:9092 --topic user-balances --property parse.key=true --property key.separator=: < /tmp/balances.txt"

---

## <span style="color: #11998e;">Step 4:</span> Inspect `.log` BEFORE Compaction

Right now, these messages are sitting in the active segment. Since Kafka never compacts the active segment, if we dump the `.log` file, we will see **all 6 messages** with consecutive offsets (0, 1, 2, 3, 4, 5).

In [ ]:
!docker exec kafka kafka-dump-log --print-data-log --files /var/lib/kafka/data/user-balances-0/00000000000000000000.log

---

## <span style="color: #11998e;">Step 5:</span> Force a Segment Roll

To make these messages eligible for compaction, we need to push them into an "inactive" segment. We set `segment.ms=100`, but Kafka only evaluates that timer when a *new* message arrives!

Let's produce one more "dummy" message. This will force Kafka to realize 100ms has passed, roll the active segment into an inactive one, and wake up the cleaner thread.

In [ ]:
!docker exec kafka bash -c "echo 'dummy:1' | kafka-console-producer --bootstrap-server localhost:9092 --topic user-balances --property parse.key=true --property key.separator=:"

*(Wait about 10 seconds before running the next step so the background cleaner thread has time to do its job).* 

---

## <span style="color: #11998e;">Step 6:</span> Inspect `.log` AFTER Compaction

Let's dump the exact same `.log` file again! This time, you should see that Kafka has physically deleted the old states. You will notice the offsets are now **Non-consecutive** (e.g., offset 0, 1, and 2 are gone) because the file was compacted!

In [ ]:
!docker exec kafka kafka-dump-log --print-data-log --files /var/lib/kafka/data/user-balances-0/00000000000000000000.log

---

## <span style="color: #11998e;">Step 7:</span> Consume Compacted Data

Finally, let's consume the topic from the beginning as a normal client. You will only see the latest balance for each user (`user1:200`, `user2:80`, `user3:50`).

In [ ]:
!docker exec kafka kafka-console-consumer \
  --bootstrap-server localhost:9092 \
  --topic user-balances \
  --from-beginning \
  --property print.key=true \
  --property key.separator=" : " \
  --timeout-ms 5000

---

## 🧹 Clean Up

Stop the cluster and remove volumes to clean up the disk space.

In [ ]:
!docker-compose -f ../../docker-compose.yml down -v

<div style="background-color: rgba(17, 153, 142, 0.1); border: 1px solid rgba(17, 153, 142, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #11998e; margin-bottom: 10px;">🎉 Lab 5 Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You've successfully demonstrated how Kafka can be used as a state store by leveraging log compaction, and inspected its effects on the physical log file!</p>
</div>